# TP2 – Genetic Algorithm (Python version)
**Goal:** maximise f(x) = -x² + 4x on the interval [1, 30].

**Fixed by the assignment**
- Population of 4 individuals (chromosomes), initial population: 18, 6, 11, 27
- Each individual is encoded on 5 bits (genes)
- Uniform crossover
- 4 scenarios to test:

| Scenario | Pc | Pm | max-generation |
|---|---|---|---|
| 1 | 0.75 | 0.005 | 30 |
| 2 | 0.75 | 0.005 | 50 |
| 3 | 0.90 | 0.01 | 30 |
| 4 | 0.90 | 0.01 | 50 |

**Statistics collected for each scenario:** execution time, fitness vector (one value per iteration), average fitness, maximum fitness, number of crossovers, number of mutations, number of accepted new solutions.

## 0. Imports

In [22]:
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Initialization and fitness function

**Objective function.** We maximise f(x) = -x² + 4x on the interval [1, 30].

**Chromosome representation.** Each solution x is encoded as a chromosome of 5 bits
(genes), because 2⁵ = 32 values can be represented (0 to 31) and the interval needs
values up to 30. Example: 18 → 10010, 6 → 00110, 11 → 01011, 27 → 11011.

**Initial population.** 4 individuals given by the assignment: 18, 6, 11, 27.

**Fitness evaluation.** The fitness of a chromosome is f(x), where x is its decoded
integer value. All four initial individuals have a negative fitness, because the
maximum of f is at x = 2 and they are far from it.

**Handling x = 0 and x = 31.** 5 bits can also represent 0 and 31, which are outside
[1, 30]. Crossover or mutation can produce them, so we use a penalty: any chromosome
outside [1, 30] gets a very bad fitness (-10⁹). It is therefore never selected and
never accepted into the population.

In [23]:
N_BITS = 5  # chromosome length
X_MIN, X_MAX = 1, 30  # valid interval for x
POP_SIZE = 4  # population size
INITIAL_POP = [18, 6, 11, 27]  # given initial population
PENALTY = -1e9  # fitness of invalid chromosomes (x = 0 or x = 31)


def encode(x):
    """integer -> list of 5 bits, e.g. 18 -> [1,0,0,1,0]"""
    return [int(b) for b in format(x, f"0{N_BITS}b")]


def decode(chrom):
    """list of 5 bits -> integer, e.g. [1,0,0,1,0] -> 18"""
    return int("".join(map(str, chrom)), 2)


def f(x):
    """objective function"""
    return -(x**2) + 4 * x


def fitness(chrom):
    """fitness of a chromosome, with a penalty outside [1, 30]"""
    x = decode(chrom)
    if x < X_MIN or x > X_MAX:
        return PENALTY
    return f(x)


def initial_population():
    return [encode(x) for x in INITIAL_POP]


# quick check
for x in INITIAL_POP:
    print(f"x = {x:2d}  chromosome = {encode(x)}  fitness = {fitness(encode(x))}")
print("x = 0 ->", fitness(encode(0)), "| x = 31 ->", fitness(encode(31)))


x = 18  chromosome = [1, 0, 0, 1, 0]  fitness = -252
x =  6  chromosome = [0, 0, 1, 1, 0]  fitness = -12
x = 11  chromosome = [0, 1, 0, 1, 1]  fitness = -77
x = 27  chromosome = [1, 1, 0, 1, 1]  fitness = -621
x = 0 -> -1000000000.0 | x = 31 -> -1000000000.0


## 2. Selection

**Method: tournament selection.** To pick one parent, we draw 2 individuals at random (with replacement). The one with the better fitness wins with probability p = 0.8, otherwise the weaker one wins. We repeat this POP_SIZE = 4 times to get 4 parents.

**Why tournament.** It only compares two fitness values, so it works with the negative fitness of f(x) = -x² + 4x. It also keeps some diversity, because a weaker individual can still win.

**Implementation notes.**
- `tournament(population)` returns one winner. `selection(population)` calls it 4 times.
- The winner is returned as a copy (`winner[:]`), so crossover and mutation never modify the old population.
- p is the same in all 4 scenarios.

In [24]:
TOURNAMENT_P = 0.8  # probability that the fitter of the 2 contestants wins


def tournament(population):
    """Draw 2 individuals (with replacement), the fitter one wins with probability p."""
    a = random.choice(population)
    b = random.choice(population)
    better, worse = (a, b) if fitness(a) >= fitness(b) else (b, a)
    winner = better if random.random() < TOURNAMENT_P else worse
    return winner[:]  # copy, so later crossover/mutation don't modify the population


def selection(population):
    """Return POP_SIZE parents chosen by tournament."""
    return [tournament(population) for _ in range(POP_SIZE)]


# quick check: how often is each individual selected over 10000 draws?
from collections import Counter

pop = initial_population()
counts = Counter()
for _ in range(10000):
    for parent in selection(pop):
        counts[decode(parent)] += 1
for x in INITIAL_POP:
    print(f"x = {x:2d}  fitness = {f(x):5d}  selected {counts[x]/40000:.1%} of the time")

x = 18  fitness =  -252  selected 21.8% of the time
x =  6  fitness =   -12  selected 36.0% of the time
x = 11  fitness =   -77  selected 28.8% of the time
x = 27  fitness =  -621  selected 13.4% of the time


## 3. Uniform Crossover

**Method: Uniform crossover.** For each gene position, a random mask (0 or 1) is generated. If the mask is 0, the child takes the gene from Parent 1. If the mask is 1, it takes the gene from Parent 2. With 5 bits, there are \(2^5 = 32\) possible masks.

**Role of \(P_c\).** \(P_c\) determines the crossover frequency. If `random.random() < pc` (0.75 or 0.90), the crossover is performed. Otherwise, children are exact copies of the parents.

**Implementation notes.** A global variable `nb_croisements` is incremented each time a crossover is actually performed.




In [25]:
nb_croisements = 0

def uniform_crossover(parent1, parent2, pc):
    global nb_croisements
    
    if random.random() < pc:
        nb_croisements += 1
        
        child1 = []
        child2 = []
        
        for i in range(N_BITS):
            if random.choice([0, 1]) == 0:
                child1.append(parent1[i])
                child2.append(parent2[i])
            else:
                child1.append(parent2[i])
                child2.append(parent1[i])
                
        return child1, child2
    else:
        return parent1[:], parent2[:]

### Test de validation (Scénario 1)
Test du croisement avec les paramètres du Scénario 1 (\(P_c = 0.75\)).

In [26]:
pop = initial_population()
parents = selection(pop)
parent1 = parents[0]
parent2 = parents[1]

pc = 0.75
child1, child2 = uniform_crossover(parent1, parent2, pc)

print(f"Parent 1 : {parent1} (x = {decode(parent1):2d})")
print(f"Parent 2 : {parent2} (x = {decode(parent2):2d})")
print(f"Enfant 1 : {child1} (x = {decode(child1):2d})")
print(f"Enfant 2 : {child2} (x = {decode(child2):2d})")
print(f"Nombre de croisements effectués : {nb_croisements}")

Parent 1 : [0, 0, 1, 1, 0] (x =  6)
Parent 2 : [1, 0, 0, 1, 0] (x = 18)
Enfant 1 : [0, 0, 0, 1, 0] (x =  2)
Enfant 2 : [1, 0, 1, 1, 0] (x = 22)
Nombre de croisements effectués : 1
